# Checkpoint 02 - fluxo contínuo via SSE

Este checkpoint mantém a comunicação REST apresentada no **Checkpoint 01** e acrescenta o acompanhamento contínuo do Firebase com **Server-Sent Events (SSE)**.

Assim, não precisamos consultar o banco repetidamente em intervalos fixos.

## 1. Configuração

In [ ]:
# CONFIGURAÇÃO DA OFICINA
# Use a mesma URL do Firebase Realtime Database fornecida para a oficina.
FIREBASE_URL = "https://oficina-python-saude-teste-default-rtdb.firebaseio.com"  # Endereço base do Realtime Database usado na oficina.
STREAM_PATH = "oficina/live"  # Caminho do nó que contém sempre o pacote mais recente.

# Parâmetros da oficina
FS = 250  # Frequência de amostragem dos sinais: 250 amostras por segundo.
AMOSTRAS_POR_PACOTE = 50  # Quantidade esperada de amostras em cada pacote publicado.
JANELA_S = 8  # Duração, em segundos, da janela de sinal mantida no buffer.

## 2. Reutilização do Checkpoint 01

A URL REST e a leitura HTTP simples continuam disponíveis.  
O `requests` permanece sendo a biblioteca responsável pela comunicação com o Firebase.

In [ ]:
import requests                     # Abre a conexão HTTP contínua com o Firebase.

# Função: Monta a URL REST completa do nó do Firebase, evitando barras duplicadas.
def firebase_node_url(firebase_url, node=STREAM_PATH):
    base = firebase_url.rstrip("/")  # Remove uma barra final para evitar "//" ao montar a URL.
    node = node.strip("/")  # Remove barras no início/fim do caminho antes de juntá-lo à URL.
    return f"{base}/{node}.json"  # O sufixo .json permite acessar esse nó pela API REST do Firebase.

# Função: Faz uma leitura única do pacote atualmente armazenado no nó da oficina.
def ler_pacote_atual(firebase_url=FIREBASE_URL, node=STREAM_PATH):
    r = requests.get(firebase_node_url(firebase_url, node), timeout=10)  # Faz um GET e espera no máximo 10 s pela resposta.
    r.raise_for_status()  # Interrompe com erro se o servidor responder com código HTTP de falha.
    return r.json()  # Converte a resposta JSON do Firebase diretamente em um objeto Python.

## 3. Novidade deste checkpoint — streaming SSE

Entram agora:

- `json`, para interpretar o conteúdo dos eventos;
- `time`, para limitar a duração da captura;
- `iterar_sse()`, que separa o fluxo em eventos;
- `pacotes_firebase()`, que mantém a conexão aberta e entrega cada pacote recebido.

In [ ]:
import json                         # Converte o texto JSON recebido pelo SSE em dicionários Python.
import time                         # Controla por quanto tempo o notebook permanece acompanhando o fluxo.

# Função: Separa o fluxo SSE em eventos completos, identificando os campos event e data.
def iterar_sse(response):
    """Interpreta eventos Server-Sent Events retornados pelo Firebase REST."""
    evento = None
    dados = []

    for raw in response.iter_lines(decode_unicode=True):  # Lê a resposta contínua linha a linha, já convertida para texto.
        if raw is None:
            continue

        linha = raw.strip()  # Remove espaços/quebras de linha antes de interpretar o campo SSE.

        if linha == "":  # No SSE, uma linha vazia indica o fim de um evento.
            if evento is not None and dados:
                yield evento, "\n".join(dados)
            evento = None
            dados = []
            continue

        if linha.startswith(":"):  # Linhas iniciadas por ":" são comentários/keep-alive do protocolo SSE.
            continue

        if linha.startswith("event:"):
            evento = linha.split(":", 1)[1].strip()  # Extrai apenas o valor que vem depois de "event:".
        elif linha.startswith("data:"):
            dados.append(linha.split(":", 1)[1].lstrip())  # Guarda apenas o conteúdo que vem depois de "data:".

# Função: Mantém a conexão SSE aberta e entrega, um a um, os pacotes válidos recebidos.
def pacotes_firebase(firebase_url=FIREBASE_URL, node=STREAM_PATH, duracao_s=20):
    """Gera pacotes recebidos pelo stream SSE do Firebase."""
    url = firebase_node_url(firebase_url, node)  # Obtém a URL REST completa do nó que será acompanhado.
    headers = {"Accept": "text/event-stream"}  # Informa ao Firebase que queremos receber Server-Sent Events.
    inicio = time.time()  # Marca o instante inicial para limitar a duração da captura.

    with requests.get(url, headers=headers, stream=True, timeout=(10, 35)) as response:
        response.raise_for_status()  # Garante que a conexão foi aceita antes de processar o fluxo.
        for evento, texto in iterar_sse(response):
            if time.time() - inicio >= duracao_s:
                return
            if evento not in ("put", "patch"):
                continue

            envelope = json.loads(texto)  # Transforma o texto JSON do evento SSE em dicionário Python.
            data = envelope.get("data")  # No SSE do Firebase, o conteúdo alterado vem dentro da chave data.
            if isinstance(data, dict) and "seq" in data:
                yield data  # Entrega o pacote ao laço for sem encerrar a conexão SSE.

## 4. Executar e acompanhar a sequência dos pacotes

Além de receber continuamente, observamos `session_id`, `seq` e `sample_start`.
Um salto em `seq` indica que um ou mais pacotes podem não ter sido recebidos.

In [ ]:
ultima_sessao = None
ultimo_seq = None

for pacote in pacotes_firebase(duracao_s=20):
    sessao = str(pacote.get("session_id", ""))
    seq = int(pacote["seq"])
    sample_start = int(pacote.get("sample_start", -1))

    if sessao != ultima_sessao:
        print("\nNOVA SESSÃO:", sessao)
        ultima_sessao = sessao
        ultimo_seq = None

    salto = 0 if ultimo_seq is None else max(0, seq - ultimo_seq - 1)  # Calcula quantos números de sequência ficaram faltando.
    print(
        f"seq={seq:5d}  sample_start={sample_start:7d}  "
        f"n={pacote.get('n')}  pacotes_ausentes={salto}"
    )
    ultimo_seq = seq

print("CHECKPOINT 2 OK")